In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import from_json , col , explode_outer, from_unixtime

########################################
#########---Sales Orders-----#######
########################################

def sales_orders(df):
    sales_orders = df.select(
        "order_number",
        "customer_id",
        "customer_name",
        "number_of_line_items",
        from_unixtime(col("order_datetime")).alias("order_timestamp")
        )
    return sales_orders

########################################
#########---ORDERED PRODUCTS-----#######
########################################


def df_ordered_products(df):

    ordered_product_schema = ArrayType(
        StructType([
            StructField("curr", StringType()),
            StructField("id", StringType()),
            StructField("name", StringType()),
            StructField("price", StringType()),
            StructField("promotion_info", StringType()),
            StructField("qty",StringType()),
            StructField("unit", StringType())
        ])
    )

    df_ordered_products = df.withColumn("ordered_products", from_json(col("ordered_products"), ordered_product_schema ))\
                .withColumn("ordered_products", explode_outer("ordered_products"))\
                .select(
                    "customer_id",
                    "customer_name",
                    "order_number",
                    "ordered_products.id",
                        col("ordered_products.name").alias("product_name"),
                    "ordered_products.price",
                    "ordered_products.curr",
                    "ordered_products.qty",
                    "ordered_products.unit" )
    return df_ordered_products

########################################
#########---PROMOTIONS-----#######
########################################

def promotions(df):

    promo_info_schema = ArrayType(
        StructType([
            StructField("promo_disc", StringType()),
            StructField("promo_id", StringType()),
            StructField("promo_item", StringType()),
            StructField("promo_qty", StringType())
        ])
    )

    df_promo_info = df.withColumn("promo_info", from_json(col("promo_info"), promo_info_schema ))\
                    .withColumn("promo_info", explode_outer("promo_info"))\
                    .filter(col("promo_info").isNotNull())\
                    .select(
                            "customer_id",
                            "customer_name",
                            "order_number",
                            "promo_info.promo_disc",
                            "promo_info.promo_id",
                            "promo_info.promo_item",
                            "promo_info.promo_qty" )
    return df_promo_info

########################################
#########---PROMOTIONS-----#######
########################################

def clicked_items(df):
    clicked_item_schema = ArrayType(
        ArrayType(StringType())
    )

    clicked_items = df.withColumn("clicked_items", from_json(col("clicked_items"), clicked_item_schema))\
                     .withColumn("clicked_items",explode_outer("clicked_items"))\
                    .select(
                        "customer_id",
                        "customer_name",
                        "order_number",
                        col("clicked_items")[0].alias("product_id"),
                        col("clicked_items")[1].alias("score")
                        )
    return clicked_items

############################
######---MAIN CODE---#######
############################

print(f"Reading source table")
df = spark.read.table("ecommerce_analytics.bronze.sales_orders")

print(f"Transforming data - Processing sales_orders Table ")
sales_orders = sales_orders(df)
print(f"Writing data - Writing sales_orders Table")
sales_orders.write.mode("overwrite").saveAsTable("ecommerce_analytics.silver.sales_orders")

print(f"Transforming data - Processing Ordered Products Table ")
df_ordered_products = df_ordered_products(df)
print(f"Writing data - Writing Odered Products Table")
df_ordered_products.write.mode("overwrite").saveAsTable("ecommerce_analytics.silver.ordered_products")


print(f"Transforming data - Processing Promotions Table ")
df_promo_info = promotions(df)
print(f"Writing data - Writing Promotions Table")
df_promo_info.write.mode("overwrite").saveAsTable("ecommerce_analytics.silver.promotions")


print(f"Transforming data - Processing clicked_items Table ")
clicked_items = clicked_items(df)
print(f"Writing data - Writing clicked_items Table")
clicked_items.write.mode("overwrite").saveAsTable("ecommerce_analytics.silver.clicked_items")

print(f"Successfully wrote all tables")